# Part 1: Data Exploration and Preprocessing

In this notebook, you will implement functions to load, preprocess, and visualize physiological data from the Wearable Exam Stress Dataset.

In [37]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import os

# Set style for plots
plt.style.use('seaborn-v0_8')
%matplotlib inline

os.getcwd()

'/Users/hannahlusk/env_folder/4-it-s-about-time-luskhannah'

## 1. Data Loading

Implement the `load_data` function to read and organize the physiological data from the dataset.

In [38]:
def load_data(data_dir='data/raw', sampling_rate_hz=1):
    """
    Loads HR, EDA, and TEMP data with derived timestamps.
    Assumes first row is the starting timestamp (Unix seconds),
    and remaining rows are signal values sampled at a fixed interval.
    """
    signal_map = {
        'hr': 'heart_rate',
        'eda': 'eda',
        'temp': 'temperature',
    }

    all_data = []

    for subject in os.listdir(data_dir):
        subject_path = os.path.join(data_dir, subject)
        if not os.path.isdir(subject_path):
            continue

        for session in os.listdir(subject_path):
            session_path = os.path.join(subject_path, session)
            if not os.path.isdir(session_path):
                continue

            signal_frames = {}

            for file in os.listdir(session_path):
                name, ext = os.path.splitext(file)
                if ext.lower() != '.csv':
                    continue
                key = name.lower()
                if key in signal_map:
                    file_path = os.path.join(session_path, file)
                    try:
                        values = pd.read_csv(file_path, header=None).squeeze()
                        if values.empty or len(values) < 2:
                            print(f"Skipping {file_path} (not enough data)")
                            continue
                        start_time = pd.to_datetime(values.iloc[0], unit='s')
                        signal_series = values.iloc[1:].reset_index(drop=True)
                        timestamps = pd.date_range(start=start_time, periods=len(signal_series), freq=f'{int(1000 / sampling_rate_hz)}ms')
                        df = pd.DataFrame({
                            'timestamp': timestamps,
                            signal_map[key]: signal_series
                        })
                        signal_frames[signal_map[key]] = df
                    except Exception as e:
                        print(f"Error reading {file_path}: {e}")

            if not signal_frames:
                print(f"No valid signals found in {subject}/{session}")
                continue

            # Merge signals on timestamp
            merged = None
            for signal_df in signal_frames.values():
                if merged is None:
                    merged = signal_df
                else:
                    merged = pd.merge(merged, signal_df, on='timestamp', how='outer')

            merged['subject_id'] = subject
            merged['session'] = session
            all_data.append(merged)

    if not all_data:
        raise ValueError("No valid data found in the directory.")

    return pd.concat(all_data, ignore_index=True)


## 2. Data Preprocessing

Implement the `preprocess_data` function to clean and prepare the data for analysis.

In [39]:
def preprocess_data(data, output_dir='data/processed', file_format='csv'):
    """
    Clean and preprocess physiological data by subject/session.

    Parameters
    ----------
    data : pd.DataFrame
        Raw data with columns: ['timestamp', 'heart_rate', 'eda', 'temperature', 'subject_id', 'session']
    output_dir : str
        Directory to save processed data files
    file_format : str
        One of: 'csv', 'parquet', or 'feather'

    Returns
    -------
    pd.DataFrame
        Combined, cleaned DataFrame for all valid subjects
    """
    # Validate file format
    file_format = file_format.lower()
    assert file_format in ['csv', 'parquet', 'feather'], "Invalid file format."

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    if not np.issubdtype(data['timestamp'].dtype, np.datetime64):
        data['timestamp'] = pd.to_datetime(data['timestamp'])

    processed_subjects = []
    numeric_cols = ['heart_rate', 'eda', 'temperature']

    for subject, subject_df in data.groupby('subject_id'):
        print(f"\n🔄 Processing subject: {subject}")
        subject_sessions = []

        for session, session_df in subject_df.groupby('session'):
            df = session_df.copy()
            df.set_index('timestamp', inplace=True)

            # Resample only numeric columns to 1s
            df_numeric = df[numeric_cols].resample('1s').mean()
            df = df_numeric

            # Diagnostic: print % missing before cleaning
            missing_pct = df.isna().mean()
            print(f"\n📊 {subject} - {session}")
            print("Missing % per signal:")
            print(missing_pct)

            # Interpolation and gap-filling
            df.interpolate(method='time', inplace=True)
            df.fillna(method='bfill', inplace=True)
            df.fillna(method='ffill', inplace=True)

            # Outlier removal via z-score (threshold 3.5)
            for col in numeric_cols:
                if df[col].count() > 10:
                    z = np.abs(stats.zscore(df[col], nan_policy='omit'))
                    df[col] = df[col].mask(z > 3.5)

            # Re-interpolate after masking outliers
            df[numeric_cols] = df[numeric_cols].interpolate(method='time')
            df.fillna(method='bfill', inplace=True)
            df.fillna(method='ffill', inplace=True)

            # Restore subject/session labels
            df = df.reset_index()
            df['subject_id'] = subject
            df['session'] = session
            subject_sessions.append(df)

        if not subject_sessions:
            print(f"⚠️ No valid sessions for subject {subject}. Skipping.")
            continue

        # Combine sessions and save
        subject_data = pd.concat(subject_sessions, ignore_index=True)
        filename = f"{subject.lower()}_processed.{file_format}"
        output_path = Path(output_dir) / filename

        if file_format == 'csv':
            subject_data.to_csv(output_path, index=False)
        elif file_format == 'parquet':
            subject_data.to_parquet(output_path, index=False)
        elif file_format == 'feather':
            subject_data.to_feather(output_path)

        print(f"✅ Saved: {output_path}")
        processed_subjects.append(subject_data)

    if not processed_subjects:
        raise ValueError("No valid data was processed.")

    return pd.concat(processed_subjects, ignore_index=True)

_ = preprocess_data(load_data('data/raw'), file_format='csv')



🔄 Processing subject: S1

📊 S1 - Final
Missing % per signal:
heart_rate     0.750088
eda            0.000021
temperature    0.000000
dtype: float64

📊 S1 - Midterm 1
Missing % per signal:
heart_rate     0.750185
eda            0.000000
temperature    0.000000
dtype: float64

📊 S1 - Midterm 2
Missing % per signal:
heart_rate     0.750185
eda            0.000000
temperature    0.000000
dtype: float64


/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:51: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:52: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:62: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:63: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead

✅ Saved: data/processed/s1_processed.csv

🔄 Processing subject: S10

📊 S10 - Final
Missing % per signal:
heart_rate     0.750089
eda            0.000000
temperature    0.000043
dtype: float64

📊 S10 - Midterm 1
Missing % per signal:
heart_rate     0.750166
eda            0.000000
temperature    0.000043
dtype: float64

📊 S10 - Midterm 2
Missing % per signal:
heart_rate     0.750178
eda            0.000038
temperature    0.000000
dtype: float64


/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:51: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:52: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:62: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:63: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead

✅ Saved: data/processed/s10_processed.csv

🔄 Processing subject: S2

📊 S2 - Final
Missing % per signal:
heart_rate     0.750086
eda            0.000000
temperature    0.000020
dtype: float64

📊 S2 - Midterm 1
Missing % per signal:
heart_rate     0.750172
eda            0.000042
temperature    0.000000
dtype: float64

📊 S2 - Midterm 2
Missing % per signal:
heart_rate     0.750149
eda            0.000000
temperature    0.000000
dtype: float64


...

✅ Saved: data/processed/s7_processed.csv

🔄 Processing subject: S8

📊 S8 - Final
Missing % per signal:
heart_rate     0.750122
eda            0.000000
temperature    0.000028
dtype: float64

📊 S8 - Midterm 1
Missing % per signal:
heart_rate     0.750203
eda            0.000000
temperature    0.000046
dtype: float64

📊 S8 - Midterm 2
Missing % per signal:
heart_rate     0.750220
eda            0.000000
temperature    0.000151
dtype: float64


/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:51: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:52: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:62: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:63: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead

✅ Saved: data/processed/s8_processed.csv

🔄 Processing subject: S9

📊 S9 - Final
Missing % per signal:
heart_rate     0.750154
eda            0.000000
temperature    0.000035
dtype: float64

📊 S9 - Midterm 1
Missing % per signal:
heart_rate     0.750163
eda            0.000000
temperature    0.000079
dtype: float64

📊 S9 - Midterm 2
Missing % per signal:
heart_rate     0.750186
eda            0.000000
temperature    0.000161
dtype: float64


/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:51: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:52: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:62: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
/var/folders/4h/kzhxx0lj3_v2vsxlh0r31jlh0000gn/T/ipykernel_32568/3777939645.py:63: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead

✅ Saved: data/processed/s9_processed.csv


## 3. Visualization

Implement the `plot_physiological_signals` function to create visualizations of the physiological data.

In [ ]:
def plot_physiological_signals(data, subject_id, session, output_dir='plots'):
    """Create plots of physiological signals for a given subject and session.
    
    Parameters
    ----------
    data : pd.DataFrame
        Preprocessed physiological data
    subject_id : str
        Subject identifier (e.g., 'S1')
    session : str
        Session identifier (e.g., 'Midterm 1')
    output_dir : str
        Directory to save plot files
        
    Returns
    -------
    matplotlib.figure.Figure
        Figure object containing the plots
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Your code here
    # 1. Create figure with subplots
    # 2. Plot each physiological signal
    # 3. Add labels and titles
    # 4. Save plot to file
    
    pass